<a href="https://colab.research.google.com/github/Ingrid-RSilva/LAB-P2-09-Arquitetura-RAG-Avancada-HNSW-HyDE-e-Cross-Encoders-/blob/main/lab09_colab_(3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LABORATÓRIO 09 — Arquitetura RAG Avançada
## HNSW + HyDE (Groq/LLaMA 3) + Cross-Encoders

---

**Objetivo:** Construir um pipeline RAG de nível de produção para busca semântica em manuais médicos técnicos, superando a limitação da similaridade de cosseno pura quando o usuário faz perguntas em linguagem coloquial.

**Problema:** A query *"dor de cabeça latejante e luz incomodando"* está geometricamente longe de *"cefaleia pulsátil com fotofobia"* no espaço vetorial, mesmo sendo semanticamente equivalente.

**Solução:**
```
Query coloquial → [HyDE/LLaMA 3] → Doc. Hipotético → [HNSW] → Top-10 → [Cross-Encoder] → Top-3
```

---
| Passo | Técnica | Função |
|:---:|---|---|
| 1 | **HNSW** (FAISS) | Indexar o corpus como grafo hierárquico |
| 2 | **HyDE** via Groq/LLaMA 3 | Transformar query coloquial em jargão técnico |
| 3 | **Bi-Encoder** | Busca rápida → Top-10 candidatos |
| 4 | **Cross-Encoder** | Re-ranking preciso → Top-3 finais |

---
## Instalação das Dependências

> **Execute esta célula primeiro e aguarde a instalação terminar.**

In [8]:
!pip install -q faiss-cpu sentence-transformers groq
print(" Dependências instaladas com sucesso!")

 Dependências instaladas com sucesso!


---
## Configuração da Chave da API (Groq)

A chave é lida com segurança pelo **Secrets do Colab** — ela nunca aparece no código.

**Como configurar:**
1. No menu lateral esquerdo, clique no ícone de **cadeado (Secrets)**
2. Clique em **"Add new secret"**
3. **Name:** `GROQ_API_KEY` · **Value:** sua chave do Groq
4. Ative a chave para este notebook e execute a célula abaixo

> Obtenha sua chave gratuita em [console.groq.com](https://console.groq.com)

In [9]:
from google.colab import userdata
from groq import Groq

GROQ_API_KEY = userdata.get("GROQ_API_KEY")
groq_client = Groq(api_key=GROQ_API_KEY)

print(" Chave Groq carregada com sucesso! (via Secrets — nenhuma chave exposta no código)")

 Chave Groq carregada com sucesso! (via Secrets — nenhuma chave exposta no código)



## Imports e Configurações Globais

In [14]:
import numpy as np
from typing import List, Dict
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss

# Modelos
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
GROQ_MODEL = "llama-3.3-70b-versatile"

# Hiperparâmetros do funil de recuperação
TOP_K_RETRIEVE = 10 # documentos recuperados pelo Bi-Encoder
TOP_K_RERANK = 3 # documentos finais após Cross-Encoder

print(" Imports OK")
print(f" Bi-Encoder : {EMBEDDING_MODEL}")
print(f" Cross-Encoder: {CROSS_ENCODER_MODEL}")
print(f" LLM (HyDE) : {GROQ_MODEL} via Groq")
print(f" Funil : Top-{TOP_K_RETRIEVE} → Top-{TOP_K_RERANK}")

 Imports OK
 Bi-Encoder : sentence-transformers/all-MiniLM-L6-v2
 Cross-Encoder: cross-encoder/ms-marco-MiniLM-L-6-v2
 LLM (HyDE) : llama-3.3-70b-versatile via Groq
 Funil : Top-10 → Top-3



## PASSO 0 — Corpus Simulado

22 fragmentos de manuais médicos técnicos cobrindo neurologia, farmacologia, emergências e protocolos clínicos.

In [15]:
MEDICAL_CORPUS: List[Dict] = [
 {"id": 0, "title": "Manual de Neurologia Clínica – Cap. 3",
 "text": "Cefaleia pulsátil acompanhada de fotofobia e fonofobia é o quadro clássico da enxaqueca (migrânea) sem aura. A dor localiza-se unilateralmente na região frontotemporal e pode durar entre 4 e 72 horas. O tratamento agudo de primeira linha inclui analgésicos AINEs e triptanos."},
 {"id": 1, "title": "Protocolo de Urgências Neurológicas",
 "text": "Cefaleia em trovoada (thunderclap headache) de início súbito e intensidade máxima em menos de 60 segundos deve ser considerada hemorragia subaracnoidea até prova em contrário. Solicitar TC de crânio sem contraste imediatamente."},
 {"id": 2, "title": "Manual de Oftalmologia – Sintomas Associados",
 "text": "Fotofobia, definida como hipersensibilidade dolorosa à luz, pode ser sintoma de meningite, cefaleia em salvas, uveíte anterior ou enxaqueca. A avaliação diferencial inclui inspeção do reflexo pupilar e avaliação de rigidez de nuca."},
 {"id": 3, "title": "Farmacologia Clínica – Analgésicos",
 "text": "Os triptanos (sumatriptana, rizatriptana) agem como agonistas seletivos dos receptores 5-HT1B/1D, promovendo vasoconstrição das artérias intracranianas e inibição da liberação de neuropeptídeos pró-inflamatórios. Contraindicados em doença cardiovascular isquêmica."},
 {"id": 4, "title": "Semiologia Médica – Avaliação da Dor",
 "text": "A escala visual analógica (EVA) quantifica a intensidade da dor de 0 a 10. Dores pulsáteis ou latejantes correlacionam-se frequentemente com mecanismo vascular, enquanto dores constritivas ou em pressão sugerem cefaleia tensional."},
 {"id": 5, "title": "Manual de Neurologia – Cefaleia Tensional",
 "text": "A cefaleia tensional episódica caracteriza-se por pressão bilateral, não pulsátil, de intensidade leve a moderada, sem náuseas e sem agravamento pela atividade física rotineira. É o tipo de cefaleia mais prevalente na população geral."},
 {"id": 6, "title": "Protocolo Clínico – Meningite Bacteriana",
 "text": "A tríade clássica da meningite bacteriana aguda inclui febre alta, rigidez de nuca (meningismo) e alteração do nível de consciência. Cefaleia intensa, fotofobia e vômitos em jato completam o quadro. Punção lombar é obrigatória na ausência de contraindicações neurológicas."},
 {"id": 7, "title": "Diretriz de Hipertensão Arterial Sistêmica",
 "text": "A crise hipertensiva (PA > 180/120 mmHg) pode manifestar-se com cefaleia occipital pulsátil, epistaxe, visão turva e dispneia. A emergência hipertensiva ocorre quando há lesão de órgão-alvo: encefalopatia, infarto agudo ou dissecção aórtica."},
 {"id": 8, "title": "Manual de Otorrinolaringologia – Sinusite",
 "text": "A sinusite maxilar aguda bacteriana causa dor facial de caráter pressivo na região malar e frontal, agravada pela posição ortostática. Cefaleia frontal intensa ao inclinar a cabeça para frente é sinal característico de sinusite frontal."},
 {"id": 9, "title": "Neurologia – Cefaleia em Salvas (Cluster Headache)",
 "text": "A cefaleia em salvas é uma forma de cefaleia trigeminoautonômica caracterizada por dor unilateral periorbital de intensidade excruciante, com duração de 15 a 180 minutos. Sintomas autonômicos ipsilaterais: lacrimejamento, injeção conjuntival, ptose e rinorreia."},
 {"id": 10, "title": "Manual de Pediatria – Cefaleia na Infância",
 "text": "Crianças com cefaleia recorrente devem ser avaliadas para enxaqueca pediátrica, que frequentemente se apresenta de forma bilateral, com episódios mais curtos (1-72 horas). Dor abdominal e cinetose são comorbidades comuns nesta faixa etária."},
 {"id": 11, "title": "Tratado de Medicina Interna – Tireoidopatias",
 "text": "O hipotireoidismo pode causar cefaleia crônica difusa, fadiga intensa, ganho de peso, bradicardia e intolerância ao frio. O diagnóstico baseia-se na dosagem sérica de TSH elevado e T4 livre reduzido. Tratamento com levotiroxina sódica."},
 {"id": 12, "title": "Protocolo de Neuroimagem – Indicações de TC",
 "text": "Indicações absolutas de TC de crânio em cefaleia: início súbito ('pior dor da vida'), cefaleia progressiva sem melhora, associação com febre e rigidez de nuca, déficit neurológico focal, papiledema ao fundo de olho ou pós-trauma craniano."},
 {"id": 13, "title": "Manual de Anestesiologia – Dor Neuropática",
 "text": "A dor neuropática caracteriza-se por sensação de queimação, choque elétrico ou formigamento ao longo de um dermátomo. A neuralgia do trigêmeo provoca dores faciais lancinantes unilaterais, geralmente desencadeadas por estímulos táteis leves (allodynia)."},
 {"id": 14, "title": "Guia de Psiquiatria Clínica – Transtornos de Ansiedade",
 "text": "O transtorno de ansiedade generalizada (TAG) frequentemente cursa com cefaleia tensional crônica, insônia, tensão muscular cervical, irritabilidade e dificuldade de concentração. O tratamento inclui TCC e inibidores seletivos de recaptação de serotonina (ISRS)."},
 {"id": 15, "title": "Semiologia – Exame Neurológico Básico",
 "text": "O exame neurológico sumário inclui: avaliação do nível de consciência pela Escala de Glasgow, pares cranianos (II a XII), força motora dos quatro membros, reflexos tendinosos profundos, sensibilidade superficial e profunda e coordenação cerebelar."},
 {"id": 16, "title": "Manual de Geriatria – Cefaleia no Idoso",
 "text": "Arterite de células gigantes (arterite temporal) deve ser suspeitada em pacientes acima de 50 anos com cefaleia temporal nova, claudicação de mandíbula e VHS elevada. Risco de cegueira por oclusão da artéria oftálmica. Iniciar corticoide empiricamente."},
 {"id": 17, "title": "Farmacologia – Analgesia Escalonada (OMS)",
 "text": "A escada analgésica da OMS propõe: degrau 1 (dipirona, paracetamol, AINEs para dor leve), degrau 2 (tramadol para dor moderada) e degrau 3 (morfina para dor intensa). Adjuvantes como antidepressivos tricíclicos podem ser acrescentados em qualquer degrau."},
 {"id": 18, "title": "Protocolo de Pós-Operatório Neurocirúrgico",
 "text": "Cefaleia pós-raquianestesia (cefaleia postural) é consequência do vazamento de LCR pelo orifício de punção. Caracteriza-se por dor intensa ao sentar, aliviada em decúbito dorsal. Tratamento: repouso, hidratação, cafeína oral ou blood patch epidural."},
 {"id": 19, "title": "Diretriz de Cefaleia Crônica Diária",
 "text": "Cefaleia crônica diária é definida como dor de cabeça presente em mais de 15 dias por mês durante pelo menos 3 meses. Causas: enxaqueca crônica, cefaleia por uso excessivo de medicamentos (rebote analgésico), cefaleia tensional crônica e hemicrania contínua."},
 {"id": 20, "title": "Manual de Emergências – Hipertensão Intracraniana",
 "text": "Síndrome de hipertensão intracraniana: cefaleia progressiva matinal, vômitos em jato sem náusea prévia, papiledema bilateral e comprometimento progressivo da consciência. Causas: tumor cerebral, hematoma subdural, abscesso encefálico e hidrocefalia obstrutiva."},
 {"id": 21, "title": "Protocolo de AVC – Janela Terapêutica",
 "text": "O AVC isquêmico agudo pode ser tratado com trombólise intravenosa (alteplase 0,9 mg/kg) dentro de uma janela terapêutica de até 4,5 horas do início dos sintomas. A trombectomia mecânica é indicada para oclusões de grandes vasos até 24 horas."},
]

print(f" Corpus carregado: {len(MEDICAL_CORPUS)} fragmentos de manuais médicos.")

 Corpus carregado: 22 fragmentos de manuais médicos.



## PASSO 1 — Construção do Índice HNSW com FAISS

### Por que HNSW e não KNN exato?

O **KNN exato** compara a query contra **todos os N vetores** → complexidade `O(N × d)`. Para 1 milhão de documentos, isso significa ~500 ms por query.

O **HNSW** constrói um **grafo multicamada** e percorre do topo (visão macro) até a base (precisão fina) em `O(log N)` → **< 1 ms** por query.

### Hiperparâmetros e RAM

| Parâmetro | Papel | Valor |
|---|---|:---:|
| **`M`** | Conexões por nó. Maior M = mais recall, mais RAM | `32` |
| **`ef_construction`** | Fila na indexação. Maior = grafo melhor, build mais lento | `200` |
| **`ef_search`** | Fila na busca. Ajustável sem re-indexar | `50` |

**Fórmula de RAM:**
```
RAM ≈ N × d × 4 bytes (vetores) + N × M × 2 × 8 bytes (ponteiros do grafo)
```
O HNSW usa ~15–20% mais RAM que apenas os vetores brutos, mas elimina o custo `O(N)` de busca.

In [16]:
print(" Carregando modelo de embedding (Bi-Encoder)...")
bi_encoder = SentenceTransformer(EMBEDDING_MODEL)
print(f" Modelo '{EMBEDDING_MODEL}' carregado.")

print("\n Gerando embeddings do corpus...")
texts = [doc["text"] for doc in MEDICAL_CORPUS]
embeddings = bi_encoder.encode(
 texts, show_progress_bar=True, convert_to_numpy=True
).astype("float32")

# Normalizar L2 → Inner Product equivale a Cosine Similarity
faiss.normalize_L2(embeddings)

dim = embeddings.shape[1]
M = 32
ef_construction = 200
ef_search = 50

index = faiss.IndexHNSWFlat(dim, M)
index.hnsw.efConstruction = ef_construction
index.hnsw.efSearch = ef_search
index.add(embeddings)

ram_vetores = index.ntotal * dim * 4
ram_ponteiros = index.ntotal * M * 2 * 8
ram_total = ram_vetores + ram_ponteiros

print(f"\n{'='*55}")
print(f" ÍNDICE HNSW CRIADO")
print(f"{'='*55}")
print(f" Vetores indexados : {index.ntotal}")
print(f" Dimensão : {dim}")
print(f" M : {M}")
print(f" ef_construction : {ef_construction}")
print(f" ef_search : {ef_search}")
print(f" RAM estimada : {ram_total:,} bytes ({ram_total/1024:.1f} KB)")

 Carregando modelo de embedding (Bi-Encoder)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Modelo 'sentence-transformers/all-MiniLM-L6-v2' carregado.

 Gerando embeddings do corpus...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


 ÍNDICE HNSW CRIADO
 Vetores indexados : 22
 Dimensão : 384
 M : 32
 ef_construction : 200
 ef_search : 50
 RAM estimada : 45,056 bytes (44.0 KB)



## PASSO 2 — HyDE: Hypothetical Document Embeddings (Groq / LLaMA 3)

### O problema geométrico
```
 Região técnica: Região coloquial:
 "cefaleia pulsátil" "dor de cabeça latejante"
 "fotofobia" ←distância grande→
 "migrânea ICHD-3"
```

### A solução HyDE
O **LLaMA 3** (via Groq) gera uma resposta técnica **hipotética** — pode ser imprecisa, mas estará no mesmo espaço semântico dos documentos reais, servindo como âncora geométrica para a busca.

> A chave da API é lida dos **Secrets do Colab** — nunca exposta no código.

In [17]:
def generate_hypothetical_document(query: str) -> str:
 """
 Chama o LLaMA 3 via Groq API para gerar um documento hipotético técnico
 a partir de uma query coloquial do usuário (técnica HyDE).
 A chave da API é lida com segurança via Secrets do Colab.
 """
 response = groq_client.chat.completions.create(
 model=GROQ_MODEL,
 messages=[
 {
 "role": "system",
 "content": (
 "Você é um médico especialista. Dado um sintoma descrito pelo paciente "
 "em linguagem coloquial, gere um parágrafo técnico médico como se fosse "
 "um trecho de manual clínico, usando jargão médico especializado. "
 "Responda APENAS com o parágrafo técnico, sem introduções ou explicações."
 ),
 },
 {"role": "user", "content": query},
 ],
 )
 return response.choices[0].message.content.strip()


# Definir a query e gerar o documento hipotético
USER_QUERY = "dor de cabeça latejante e luz incomodando"

print(" Chamando Groq/LLaMA 3 para gerar documento hipotético...")
hypothetical_doc = generate_hypothetical_document(USER_QUERY)

print("\n HyDE — Transformação da Query")
print(f"\n Query coloquial do usuário:")
print(f" '{USER_QUERY}'")
print(f"\n Documento hipotético gerado pelo LLaMA 3:")
print(f" '{hypothetical_doc}'")

 Chamando Groq/LLaMA 3 para gerar documento hipotético...

 HyDE — Transformação da Query

 Query coloquial do usuário:
 'dor de cabeça latejante e luz incomodando'

 Documento hipotético gerado pelo LLaMA 3:
 'A cefaleia pulsátil apresentada pelo paciente sugere uma possible etiologia vascular, caracterizada por uma sensação de dor latejante e unilateral, frequentemente acompanhada de fotofobia e, em alguns casos, fonofobia. A dor é geralmente descrita como uma sensação de pulsação sincronizada com o batimento cardíaco, o que pode indicar uma possível disfunção vascular, como a migraña, que afeta cerca de 15% da população geral. Além disso, a presença de hipersensibilidade à luz pode ser um indicador de uma possível alteração na regulação da resposta ao estímulo visual, envolvendo a via aferente da retina e a formação reticular do tronco cerebral, que desempenha um papel fundamental na modulação da percepção e resposta ao estímulo doloroso. Portanto, é essencial realizar uma avaliação


## PASSO 3 — Busca Rápida via Bi-Encoder + HNSW (Top-10)

O vetor do **documento hipotético** (não da query original) é usado para buscar os documentos mais próximos no índice HNSW — o **funil largo**, priorizando alto recall.

In [18]:
# Vetorizar o documento hipotético
query_vec = bi_encoder.encode(
 [hypothetical_doc], convert_to_numpy=True
).astype("float32")
faiss.normalize_L2(query_vec)

# Buscar Top-K no índice HNSW
distances, indices_found = index.search(query_vec, TOP_K_RETRIEVE)

# Montar lista de candidatos
candidates = []
for rank, (dist, idx) in enumerate(zip(distances[0], indices_found[0]), 1):
 doc = MEDICAL_CORPUS[idx].copy()
 doc["bi_encoder_score"] = float(dist)
 doc["bi_encoder_rank"] = rank
 candidates.append(doc)

print(f" Top-{TOP_K_RETRIEVE} documentos recuperados pelo Bi-Encoder + HNSW")
print(f" (vetor do documento hipotético → índice HNSW)")
print()
print(f" {'Rank':<5} {'Score':>8} Título")
print(" " + "" * 65)
for doc in candidates:
 print(f" {doc['bi_encoder_rank']:<5} {doc['bi_encoder_score']:>8.4f} {doc['title']}")

 Top-10 documentos recuperados pelo Bi-Encoder + HNSW
 (vetor do documento hipotético → índice HNSW)

 Rank     Score Título
 
 1       0.6890 Manual de Oftalmologia – Sintomas Associados
 2       0.9865 Manual de Neurologia Clínica – Cap. 3
 3       1.0108 Semiologia Médica – Avaliação da Dor
 4       1.0163 Manual de Geriatria – Cefaleia no Idoso
 5       1.0357 Neurologia – Cefaleia em Salvas (Cluster Headache)
 6       1.0516 Protocolo de Neuroimagem – Indicações de TC
 7       1.1072 Diretriz de Hipertensão Arterial Sistêmica
 8       1.1074 Protocolo de Pós-Operatório Neurocirúrgico
 9       1.1487 Protocolo de Urgências Neurológicas
 10      1.1596 Tratado de Medicina Interna – Tireoidopatias


---
## PASSO 4 — Re-ranking com Cross-Encoder (Top-3 Finais)

| | Bi-Encoder | Cross-Encoder |
|---|---|---|
| **Input** | Query e Doc separados | `[CLS] Query [SEP] Doc` juntos |
| **Atenção cruzada** | Não tem | Bidirecional completa |
| **Velocidade** | Muito rápida | Lenta (par a par) |
| **Precisão** | ~80–90% recall | ~95–99% precision |
| **Uso** | Funil largo (Top-10) | Funil fino (Top-3) |

In [19]:
print(" Carregando Cross-Encoder...")
cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL)
print(f" Cross-Encoder '{CROSS_ENCODER_MODEL}' carregado.")

 Carregando Cross-Encoder...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

 Cross-Encoder 'cross-encoder/ms-marco-MiniLM-L-6-v2' carregado.


In [20]:
# Criar pares (query ORIGINAL, documento)
# IMPORTANTE: usamos a QUERY ORIGINAL, não o documento hipotético!
pairs = [(USER_QUERY, doc["text"]) for doc in candidates]

print(" Calculando scores do Cross-Encoder...")
ce_scores = cross_encoder.predict(pairs)

for doc, score in zip(candidates, ce_scores):
 doc["cross_encoder_score"] = float(score)

reranked = sorted(candidates, key=lambda d: d["cross_encoder_score"], reverse=True)

print(f"\n Ranking completo após Cross-Encoder ( = selecionados para o LLM)")
print()
print(f" {'':2} {'Rank':<5} {'CE Score':>10} {'BI Score':>10} Título")
print(" " + "" * 78)
for rank, doc in enumerate(reranked, 1):
 marker = "" if rank <= TOP_K_RERANK else " "
 print(
 f" {marker} {rank:<4} {doc['cross_encoder_score']:>10.4f} "
 f"{doc['bi_encoder_score']:>10.4f} {doc['title']}"
 )

 Calculando scores do Cross-Encoder...

 Ranking completo após Cross-Encoder ( = selecionados para o LLM)

    Rank    CE Score   BI Score Título
 
  1       -3.6563     1.0108 Semiologia Médica – Avaliação da Dor
  2       -4.9316     0.6890 Manual de Oftalmologia – Sintomas Associados
  3       -6.7814     0.9865 Manual de Neurologia Clínica – Cap. 3
   4       -8.4564     1.1074 Protocolo de Pós-Operatório Neurocirúrgico
   5       -8.9419     1.0357 Neurologia – Cefaleia em Salvas (Cluster Headache)
   6       -9.0961     1.0516 Protocolo de Neuroimagem – Indicações de TC
   7      -10.6981     1.0163 Manual de Geriatria – Cefaleia no Idoso
   8      -10.8874     1.1596 Tratado de Medicina Interna – Tireoidopatias
   9      -11.2470     1.1072 Diretriz de Hipertensão Arterial Sistêmica
   10     -11.2850     1.1487 Protocolo de Urgências Neurológicas


In [22]:
top_docs = reranked[:TOP_K_RERANK]

print(f" DOCUMENTOS FINAIS INJETADOS NO CONTEXTO DO LLM (Top-{TOP_K_RERANK})")

for rank, doc in enumerate(top_docs, 1):
 print(f"\n [{rank}] {doc['title']}")
 print(f" Cross-Encoder Score : {doc['cross_encoder_score']:.4f}")
 print(f" Bi-Encoder Score : {doc['bi_encoder_score']:.4f}")
 print(f" Texto:\n {doc['text']}")

 DOCUMENTOS FINAIS INJETADOS NO CONTEXTO DO LLM (Top-3)

 [1] Semiologia Médica – Avaliação da Dor
 Cross-Encoder Score : -3.6563
 Bi-Encoder Score : 1.0108
 Texto:
 A escala visual analógica (EVA) quantifica a intensidade da dor de 0 a 10. Dores pulsáteis ou latejantes correlacionam-se frequentemente com mecanismo vascular, enquanto dores constritivas ou em pressão sugerem cefaleia tensional.

 [2] Manual de Oftalmologia – Sintomas Associados
 Cross-Encoder Score : -4.9316
 Bi-Encoder Score : 0.6890
 Texto:
 Fotofobia, definida como hipersensibilidade dolorosa à luz, pode ser sintoma de meningite, cefaleia em salvas, uveíte anterior ou enxaqueca. A avaliação diferencial inclui inspeção do reflexo pupilar e avaliação de rigidez de nuca.

 [3] Manual de Neurologia Clínica – Cap. 3
 Cross-Encoder Score : -6.7814
 Bi-Encoder Score : 0.9865
 Texto:
 Cefaleia pulsátil acompanhada de fotofobia e fonofobia é o quadro clássico da enxaqueca (migrânea) sem aura. A dor localiza-se unilateralmente


## BÔNUS — Testar com Múltiplas Queries

Execute o pipeline completo para outras queries coloquiais usando o LLaMA 3 real.

In [25]:
def run_full_pipeline(query: str):
    print(f"  QUERY: '{query}'")
    print("  Gerando documento hipotetico via LLaMA 3...")
    hyp_doc = generate_hypothetical_document(query)
    print(f"  Doc. hipotetico: '{hyp_doc[:120]}...'")

    q_vec = bi_encoder.encode([hyp_doc], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_vec)
    dists, idxs = index.search(q_vec, TOP_K_RETRIEVE)

    cands = []
    for r, (d, i) in enumerate(zip(dists[0], idxs[0]), 1):
        doc = MEDICAL_CORPUS[i].copy()
        doc["bi_encoder_score"] = float(d)
        cands.append(doc)

    pairs = [(query, doc["text"]) for doc in cands]
    scores = cross_encoder.predict(pairs)
    for doc, s in zip(cands, scores):
        doc["cross_encoder_score"] = float(s)

    reranked = sorted(cands, key=lambda d: d["cross_encoder_score"], reverse=True)

    print(f"\n  Top-{TOP_K_RERANK} documentos selecionados:")
    for rank, doc in enumerate(reranked[:TOP_K_RERANK], 1):
        print(f"  [{rank}] (CE={doc['cross_encoder_score']:.3f}) {doc['title']}")


outras_queries = [
    "pressao alta dor na cabeca",
    "dor de cabeca todo dia ha meses",
    "tontura e vomito com ouvido tampado",
]

for q in outras_queries:
    run_full_pipeline(q)


  QUERY: 'pressao alta dor na cabeca'
  Gerando documento hipotetico via LLaMA 3...
  Doc. hipotetico: 'A hipertensão arterial pode desencadear cefaleias devido à vasodilatação e ao aumento da pressão intracraniana, caracter...'

  Top-3 documentos selecionados:
  [1] (CE=0.919) Diretriz de Cefaleia Crônica Diária
  [2] (CE=-2.663) Semiologia Médica – Avaliação da Dor
  [3] (CE=-2.889) Manual de Neurologia – Cefaleia Tensional

  QUERY: 'dor de cabeca todo dia ha meses'
  Gerando documento hipotetico via LLaMA 3...
  Doc. hipotetico: 'A cefaleia crônica diária é caracterizada por uma dor de cabeça persistente e recorrente, com duração superior a 15 dias...'

  Top-3 documentos selecionados:
  [1] (CE=6.474) Diretriz de Cefaleia Crônica Diária
  [2] (CE=-6.039) Protocolo de Neuroimagem – Indicações de TC
  [3] (CE=-6.419) Manual de Anestesiologia – Dor Neuropática

  QUERY: 'tontura e vomito com ouvido tampado'
  Gerando documento hipotetico via LLaMA 3...
  Doc. hipotetico: 'A apresen

## Resumo do Pipeline

```
Usuário digita query coloquial



 PASSO 2 — HyDE (Groq / LLaMA 3)
 LLM gera resposta técnica hipotética
 → resolve a lacuna semântica

 vetor do documento hipotético


 PASSO 3 — Bi-Encoder + HNSW
 Busca O(log N) no grafo hierárquico
 → Top-10 candidatos (funil largo)

 10 documentos candidatos


 PASSO 4 — Cross-Encoder
 Atenção bidirecional (query + doc)
 → Top-3 documentos finais (funil fino)



 Contexto injetado no LLM gerador
 → Resposta final ao usuário
```

| Etapa | Técnica | Velocidade | Precisão |
|---|---|:---:|:---:|
| Transformação | HyDE (LLaMA 3 / Groq) | Rápida | Resolve lacuna semântica |
| Recuperação | Bi-Encoder + HNSW | O(log N) | Alta recall |
| Refinamento | Cross-Encoder | O(K) | Alta precision |

---
> *Partes geradas/complementadas com IA, revisadas por Ingrid.*